# Remotion Colab Render Workflow
這份筆記本專為您的 Remotion 專案設計，支援：
- 自動掛載 Google Drive
- 修正 Colab 上 `eslint-config-remotion` 的依賴衝突
- 自動建立所需的素材資料夾 (`drive_assets`)
- 輸出相容 Windows Media Player 的 yuv420p 影片
- 自動複製成果至 Google Drive

In [10]:
# Cell 0: 掛載 Drive 與 Clone 專案
from google.colab import drive
import os

drive.mount('/content/drive')

if not os.path.exists('/content/remotion'):
    !git clone https://github.com/Allen930311/remotion.git /content/remotion
else:
    %cd /content/remotion
    !git pull origin main

%cd /content/remotion

# 確保 drive_assets 選項目錄存在 (根據您的 auto-skill 經驗回報)
drive_assets_path = '/content/drive/MyDrive/remotion_assets'
os.makedirs(drive_assets_path, exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '02_Audio'), exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '03_Images'), exist_ok=True)
print("環境與專案取得完成！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/remotion
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 23 (delta 7), reused 23 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 9.13 KiB | 779.00 KiB/s, done.
From https://github.com/Allen930311/remotion
 * branch            main       -> FETCH_HEAD
   e669187..5a4d981  main       -> origin/main
Updating e669187..5a4d981
Fast-forward
 diary/2026/03/2026-03-03-remotion.md               | 28 +++++++
 modify_notebook.py                                 | 43 -----------
 package-lock.json                                  |  7 ++
 package.json                                       |  3 +-
 .../geopolitics-explainer/src/GeopoliticsVideo.tsx | 34 ++++++++
 .../src/components/AnimatedChart.tsx               | 44 +++++++++++
 projects/geopoli

In [11]:
# Cell 1: 依賴修復與安裝
%cd /content/remotion
import json

package_json_path = 'package.json'
with open(package_json_path, 'r', encoding='utf-8') as f:
    pkg = json.load(f)

# 移除 eslint-config-remotion 避免在 Colab 報錯
if 'devDependencies' in pkg and 'eslint-config-remotion' in pkg['devDependencies']:
    del pkg['devDependencies']['eslint-config-remotion']
    with open(package_json_path, 'w', encoding='utf-8') as f:
        json.dump(pkg, f, indent=2)
    print("[依賴修復] 已移除 eslint-config-remotion")

!npm install

/content/remotion
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 1 package, and audited 346 packages in 2s
⠹
⠹54 packages are looking for funding
⠹  run `npm fund` for details
⠹
8 high severity vulnerabilities

To address issues that do not require attention, run:
  npm audit fix

Some issues need review, and may require choosing
a different dependency.

Run `npm audit` for details.
⠸

In [12]:
# Cell 2: 渲染設定與執行
%cd /content/remotion
import os
import re
import shutil

# 設定環境變數
os.environ["ENTRY_FILE"] = "src/index.ts"  # Remotion 進入點檔案
os.environ["COMP_NAME"] = "GeopoliticsExplainer"  # Composition 名稱

# 自動從 COMP_NAME 推導輸出檔名 (CamelCase → snake_case)
comp = os.environ['COMP_NAME']
snake = re.sub(r'(?<!^)(?=[A-Z])', '_', comp).lower()
os.environ["OUTPUT_FILE"] = f"out/{snake}.mp4"
os.environ["DRIVE_FILENAME"] = f"{snake}.mp4"  # Drive 上的檔名

# 渲染指令
render_cmd = (
    f"npx remotion render {os.environ['ENTRY_FILE']} "
    f"{os.environ['COMP_NAME']} {os.environ['OUTPUT_FILE']} "
    f"--pixel-format=yuv420p --codec=h264 --concurrency=2"
)
print(f"Executing: {render_cmd}")

!{render_cmd}

# 將成品複製到 Google Drive（檔名 = 專案名稱）
output_path = f"/content/remotion/{os.environ['OUTPUT_FILE']}"
drive_dest = f"/content/drive/MyDrive/remotion_assets/{os.environ['DRIVE_FILENAME']}"

if os.path.exists(output_path):
    shutil.copy(output_path, drive_dest)
    print(f"\n✅ 影片已成功輸出並複製至 Drive: {drive_dest}")
else:
    print("\n❌ 找不到輸出的影片，渲染可能失敗。")
